# Clase 067 — Clasificación multiclase, multilabel, multioutput

Tres escenarios más allá del binario: **multiclase** (una salida, K>2 clases), **multilabel** (varias etiquetas por muestra) y **multioutput** (varias salidas, cada una multiclase). Vemos qué estrategia de sklearn usar y qué métrica elegir.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print('X', X.shape, '| clases:', np.unique(y))

## 1. Multiclase nativo vs One-vs-Rest

`SGDClassifier` soporta K>2 nativamente (sklearn usa OvR internamente). Al envolverlo explícito en `OneVsRestClassifier` vemos los **10 clasificadores binarios** en `.estimators_`.

In [ ]:
sgd = SGDClassifier(random_state=42)
sgd.fit(Xtr, ytr)
print('accuracy SGD nativo:', round(sgd.score(Xte, yte), 4))

ovr = OneVsRestClassifier(SGDClassifier(random_state=42), n_jobs=1)
ovr.fit(Xtr, ytr)
print('accuracy OvR:', round(ovr.score(Xte, yte), 4))
print('clasificadores en OvR:', len(ovr.estimators_))

assert len(ovr.estimators_) == 10

## 2. One-vs-One con SVC

`SVC` entrena internamente K·(K-1)/2 = 45 clasificadores por pares. Con `decision_function_shape='ovo'` su `decision_function` expone los 45 scores por muestra.

In [ ]:
svc = SVC(decision_function_shape='ovo', random_state=42)
svc.fit(Xtr, ytr)
df = svc.decision_function(Xte[:1])
print('accuracy SVC (OvO):', round(svc.score(Xte, yte), 4))
print('shape de decision_function:', df.shape)
print('scores esperados = 10*9/2 =', 10 * 9 // 2)

assert df.shape[1] == 45

## 3. Multilabel con KNN

Construimos `Y = [es_grande (>=7), es_impar]`: cada muestra puede tener 0, 1 o 2 etiquetas. `KNeighborsClassifier` lo soporta nativamente. Evaluamos con F1 macro y hamming loss.

In [ ]:
Y_tr = np.c_[ytr >= 7, ytr % 2 == 1]
Y_te = np.c_[yte >= 7, yte % 2 == 1]
print('Y multilabel shape:', Y_tr.shape)

knn = KNeighborsClassifier()
knn.fit(Xtr, Y_tr)
Y_pred = knn.predict(Xte)

f1_macro = f1_score(Y_te, Y_pred, average='macro')
hl = hamming_loss(Y_te, Y_pred)
print(f'F1 macro:     {f1_macro:.4f}')
print(f'hamming loss: {hl:.4f}')

assert f1_macro > 0.9 and hl < 0.1

## 4. Macro vs micro F1

`macro` promedia por etiqueta (todas pesan igual); `micro` agrega TP/FP/FN globalmente (pesa por volumen). Difieren cuando las etiquetas están desbalanceadas.

In [ ]:
f1_micro = f1_score(Y_te, Y_pred, average='micro')
print(f'F1 macro: {f1_macro:.4f}')
print(f'F1 micro: {f1_micro:.4f}')

# proporcion de positivos por etiqueta
print('positivos por etiqueta (test):', Y_te.mean(axis=0).round(3))
print('si una etiqueta fuera muy rara, macro (que la pesa igual) caeria mas que micro.')

## 5. Multioutput: denoising

Cada píxel es una **salida multiclase** (0–16 niveles de gris). Ensuciamos las imágenes con ruido uniforme y entrenamos `KNeighborsClassifier` para reconstruir la imagen limpia.

In [ ]:
rng = np.random.RandomState(42)
Xtr_noisy = Xtr + rng.randint(0, 8, Xtr.shape)
Xte_noisy = Xte + rng.randint(0, 8, Xte.shape)

knn_dn = KNeighborsClassifier()
knn_dn.fit(Xtr_noisy, Xtr.astype(int))   # target = imagen limpia, cada pixel multiclase
clean_pred = knn_dn.predict(Xte_noisy[:1])

err_noisy = np.abs(Xte_noisy[0] - Xte[0]).mean()
err_clean = np.abs(clean_pred[0] - Xte[0]).mean()
print(f'error medio ruidosa vs original:  {err_noisy:.2f}')
print(f'error medio denoised vs original: {err_clean:.2f}')
assert err_clean < err_noisy

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for a, img, t in zip(axes, [Xte_noisy[0], clean_pred[0], Xte[0]],
                     ['ruidosa', 'denoised', 'original']):
    a.imshow(img.reshape(8, 8), cmap='binary')
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()

## Ejercicios

1. **OvR vs OvO.** ¿Cuántos clasificadores entrena cada estrategia sobre 10 clases? Verificalo con `.estimators_` y con el shape de `decision_function`.
2. **Etiqueta rara.** Cambiá una etiqueta multilabel por una muy desbalanceada (ej.: `y == 0`). ¿Cuál se mueve más, el F1 macro o el micro?
3. **MultiOutputClassifier.** Envolvé un `SGDClassifier` en `MultiOutputClassifier` para el problema multilabel. Comparalo con el KNN nativo.
4. **Más denoising.** Aplicá el denoiser a 3 imágenes y mostrá las tres tríadas (ruidosa, denoised, original).

## Conclusiones

- **Multiclase**: una etiqueta con K>2 valores; sklearn envuelve con OvR (lineal en K) u OvO (K·(K-1)/2, bueno si el base escala mal con N).
- **Multilabel**: `Y` es matriz binaria; `KNeighborsClassifier`/`RandomForest` lo soportan nativo.
- La **accuracy** multilabel (subset accuracy) es demasiado estricta: usá `hamming_loss` y F1.
- **macro** pesa cada clase igual; **micro** pesa por volumen: difieren con desbalanceo.
- **Multioutput**: cada salida es multiclase (ej.: denoising, 17 niveles por píxel); `MultiOutputClassifier` entrena un modelo por columna.